# Lab 2: Living in the Wumpus World: Logical Agents with propositional logic

In this lab we will learn how to use propositional logic and knowledge bases, which you created in Lab 1 to create a simple agent to traverse the Wumpus World.

The lab includes:

- Implementing an agent with a knowledge base of the Wumpus World, can query that knowledge base for which tiles are safe to visit.

The content of this lab relates strongly to the material of Chapter 6 (Logical Agents) in the course book (Artificial Intelligence: A Modern Approach).
The enviornment we will tackle in this lab is called the Wumpus World and is covered in the course book as well (Chapter 6.2)

# Implementation of the Wumpus World

Below you will find a code implementation of the Wumpus World along with a few helpful definitions and functions.
The Wumpus World is fully implemented since this lab is about building a propositional logic knowledge base system to build an AI around.
However, since the AI will interact with the Wumpus World, being familiar with the code and functions will help you use them later.

In [88]:
#Imports
import random
import itertools

# Directions
RIGHT = 0
DOWN  = 1
LEFT  = 2
UP    = 3

# How much the agent will move if it takes a step forward for each direction
direction_step = [(1,0), (0,-1), (-1,0), (0,1)]

# Objects
AGENT   = "a"
WUMPUS  = "w"
PIT     = "p"
GOLD    = "g"

# Actions
NONE = ""
FORWARD = "forward"
TURN_LEFT = "turn_left"
TURN_RIGHT = "turn_right"
GRAB = "grab"
SHOOT = "shoot"
CLIMB = "climb"

class WumpusWorld:
    
    def __init__(self):
        self.game_over = False
        self.agent_dir = RIGHT
        self.agent_has_arrow = True
        self.agent_has_gold = False
        self.wumpus_screams = False
        self.non_start_squares = list(itertools.product([0,1,2,3],[0,1,2,3]))
        self.non_start_squares.pop(0)
        self.cave = [[[],[],[],[]],[[],[],[],[]],[[],[],[],[]],[[],[],[],[]]]
        self.cave[0][0].append(AGENT)
        self.agent_pos = (0,0)
        self.random_place(WUMPUS)
        self.random_place(GOLD)
        for square in self.non_start_squares:
            if random.random() < 0.2:
                self.cave[square[0]][square[1]].append(PIT)
        self.performance = 0
        self.last_action = NONE
    
    def random_place(self,object):
        '''
        Randomly places an object into a tile other than the start.
        Args:
            object (any): The object (typically WUMPUS or GOLD) to place
        '''
        pos = random.choice(self.non_start_squares)
        self.cave[pos[0]][pos[1]].append(object)
        
    def show(self):
        '''
        Prints the Wumpus World cave and current agent performance.
        '''
        for y in range(3,-1,-1):
            for x in range(0,4):
                [print(o,end="") for o in self.cave[x][y]]
                print("\t",end="")
            print()
        print(f"Performance: {self.performance}")

    def actuate(self, action):
        '''
        If the game is still running, the input action is performed and the game state is moved forward.
        Args:
            action (str): The action the agent should perform
        '''
        # If the game is over, nothing can be done
        if self.game_over:
            return
        
        # store the action that was performed, decrease the performance, and remove scream from memory
        self.last_action = action
        self.performance = self.performance-1
        self.wumpus_screams = False
        
        # If the NONE action is performed, do nothing
        if action == NONE:
            return
        
        # FORWARD action: Move the agent one tile in its current direction.
        # End the game and lower performance if it steps into a PIT or WUMPUS
        if action == FORWARD:
            x, y = self.agent_pos
            self.cave[x][y].remove(AGENT)
            xp, yp = direction_step[self.agent_dir]
            x = min(max(x+xp,0),3)
            y = min(max(y+yp,0),3)
            self.agent_pos = (x,y)
            if WUMPUS in self.cave[x][y] or PIT in self.cave[x][y]:
                self.game_over = True
                self.performance = self.performance-1000
            else:
                self.cave[x][y].append(AGENT)
            return
        
        # TURN_LEFT action: Change the agents direction 90 degrees counter-clockwise
        if action == TURN_LEFT:
            self.agent_dir = (self.agent_dir-1)%4
            return
        # TURN_RIGHT action: Change the agents direction 90 degrees clockwise
        if action == TURN_RIGHT:
            self.agent_dir = (self.agent_dir+1)%4
            return
        # GRAB action: If the agent is in the same tile as the gold, remove it from the cave and add it to the agent
        if action == GRAB:
            x, y = self.agent_pos
            if GOLD in self.cave[x][y]:
                self.agent_has_gold = True
                self.cave[x][y].remove(GOLD)
            return
        # SHOOT action: If the agent has the arrow,
        #    remove it, decrease performance, and, 
        #    if there is a WUMPUS infront of the agent, remove it and add scream perception 
        if action == SHOOT:
            if not self.agent_has_arrow:
                return
            self.agent_has_arrow = False
            self.performance = self.performance-10
            x, y = self.agent_pos
            if self.agent_dir == RIGHT:
                hit_squares = [(min(x+1,3),y),(min(x+2,3),y),(min(x+3,3),y)]
            elif self.agent_dir == DOWN:
                hit_squares = [(x,max(y-1,0)),(x,max(y-2,0)),(x,max(y-3,0))]
            elif self.agent_dir == LEFT:
                hit_squares = [(max(x-1,0),y),(max(x-2,0),y),(max(x-3,0),y)]
            elif self.agent_dir == UP:
                hit_squares = [(x,min(y+1,3)),(x,min(y+2,3)),(x,min(y+3,3))]
            for x, y in hit_squares:
                if WUMPUS in self.cave[x][y]:
                    self.cave[x][y].remove(WUMPUS)
                    self.wumpus_screams = True
            return
        
        # CLIMB action: If the agent is at the starting tile, end the game and add performance if the agent has the gold
        if action == CLIMB:
            if self.agent_pos == (0,0):
                if self.agent_has_gold:
                    self.performance = self.performance+1000
                self.cave[0][0].remove(AGENT)
                self.agent_pos = None
                self.agent_dir = None
                self.game_over = True
            return
    
    def perceive(self):
        '''
        Returns for each percept whether that percept was sensed after the last action.
        '''
        x,y = self.agent_pos
        stench = False
        breeze = False
        for sx,sy in adjacent_tiles(x,y):
            if WUMPUS in self.cave[sx][sy]:
                stench = True
            if PIT in self.cave[sx][sy]:
                breeze = True
        glitter = GOLD in self.cave[x][y]
        bump = False
        if self.last_action == FORWARD:
            if self.agent_dir == RIGHT and x == 3:
                bump = True
            elif self.agent_dir == DOWN and y == 0:
                bump = True
            elif self.agent_dir == LEFT and x == 0:
                bump = True
            elif self.agent_dir == UP and y == 3:
                bump = False
        scream = self.wumpus_screams
        return(stench, breeze, glitter, bump, scream)

def adjacent_tiles(x, y):
    '''
    Helper function that, given a position, returns all tiles that are adjacent to it.
    Args:
        x (int): The x coordinate of the position
        y (int): The y coordinate of the position
    '''
    tiles = []
    for sx,sy in [(x-1,y),(x+1,y),(x,y-1),(x,y+1)]:
        if sx<4 and sx>-1 and sy<4 and sy>-1:
            tiles.append((sx,sy))
    return tiles

# INSERT THE IMPLEMENTED AND TESTED CONTENT OF LAB 1 HERE


In [90]:
# Lab 1. Task 1: classes for logical sentences

class And:
    def __init__(self, A, B):
        self.A = A
        self.B = B
        
    def __repr__(self):
        return f"({self.A} & {self.B})"


class Or:
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} | {self.B})"


class Equals:
    # Logical equivalence (iff): A <-> B
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} <-> {self.B})"


class Implies:
    # Logical implication: A -> B
    def __init__(self, A, B):
        self.A = A
        self.B = B

    def __repr__(self):
        return f"({self.A} -> {self.B})"


class Not:
    # Unary operator
    def __init__(self, A):
        self.A = A

    def __repr__(self):
        return f"~({self.A})"


# Lab 1. Task 2, 3: chaining function 

def chain(op, sentences):
    """
    Given a binary operator class (And, Or, Equals, Implies)
    and a list [s1, s2, s3, ...],
    build: op(s1, op(s2, op(s3, ...))).

    Example:
        chain(And, [s1, s2, s3]) -> And(s1, And(s2, s3))
    """
    if not sentences:
        raise ValueError("Need at least one sentence")
    if len(sentences) == 1:
        return sentences[0]

    # Right-associative: s1 op (s2 op (s3 op ...))
    result = sentences[-1]
    for i in range(len(sentences) - 2, -1, -1):
        result = op(sentences[i], result)
    return result


In [91]:
# Lab 1. Task 4,5: evaluation function

def evaluate(sentence, variables):
    """
    Evaluate a PL sentence in a given model.
    Returns: True, False, or None (unknown).
    """

    # Literal values
    if sentence is True or sentence is False or sentence is None:
        return sentence

    # Variable (string)
    if isinstance(sentence, str):
        return variables.get(sentence, None)  # None if missing

    # AND
    if isinstance(sentence, And):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is False or b is False:
            return False
        if a is True and b is True:
            return True
        return None

    # OR
    if isinstance(sentence, Or):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is True or b is True:
            return True
        if a is False and b is False:
            return False
        return None

    # NOT
    if isinstance(sentence, Not):
        v = evaluate(sentence.A, variables)
        if v is True:
            return False
        if v is False:
            return True
        return None

    # IMPLIES: A -> B
    if isinstance(sentence, Implies):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)

        if a is True:
            return b            # whatever B is (True / False / None)
        if a is False:
            return True         # False -> anything is True
        # a is None
        if b is True:
            return True
        return None

    # EQUALS: A <-> B (iff)
    if isinstance(sentence, Equals):
        a = evaluate(sentence.A, variables)
        b = evaluate(sentence.B, variables)
        if a is None or b is None:
            return None
        return (a == b)

    raise TypeError(f"Cannot evaluate sentence of type {type(sentence)}")


In [92]:
# Lab 1. Task 7: query function

import itertools

# Query function
def query(query_sentence, KB, variables):
    '''
    For a given query and knowledge base, returns the models where all KB sentences are true
    in three lists depending on if the query is True, False, or None
    '''
    vars_list = list(variables)

    true_models = []
    false_models = []
    none_models = []

    # All combinations of True/False for the relevant variables
    for values in itertools.product([False, True], repeat=len(vars_list)):
        model = dict(zip(vars_list, values))

        # Check if model is valid for the KB
        valid = True
        for sentence in KB:
            val = evaluate(sentence, model)
            if val is False:   # KB sentence cannot be False
                valid = False
                break

        if not valid:
            continue

        # Evaluate the query in this valid model
        q_val = evaluate(query_sentence, model)

        if q_val is True:
            true_models.append(model)
        elif q_val is False:
            false_models.append(model)
        else:
            none_models.append(model)

    return true_models, false_models, none_models


# Using logic to understand the Wumpus World

Now that we have a working Propositional Logic, Knowledge Base, and Query system in place, we can finally start using it to create an agent.
Knowledge-based agents can come in many forms, but the most typical behavior is to dump all the relevant knowledge of the world into a knowledge base and then query the knowledge base for what action to perform next.
The percepts from the environment is then used to add new knowledge into the knowledge base to help with the next query.

Doing all of this can require very extensive knowledge bases and complex agents, which is beyond the scope of this assigment.
Instead we will use a very simple procedure for our agent:
- If the agent has the gold we know there must be safe path back to the entrance, take actions to follow that path back and then use CLIMB to finish the game.
- If there is glitter on the current tile, use the GRAB action to pick it up.
- Otherwise, query every tile for whether its safe or not.
    - If there is a safe tile that has not yet been visited, calculate a path to it and then follow that path.
        - If there is no path, calculate a path to the next unvisited safe tile.
    - If all safe tiles have been visited or are unreachable, find a path back to the start tile and then use CLIMB to finish the game.

As you can see this agent is very simple, not even considering using the SHOOT action, and assumes that all tiles that it doesn't know for sure to be unsafe.
For now though, this is enough.
Before we can start creating an agent, we need a knowledge base for it to query.
There are many things we know about the Wumpus World, many of which we could turn into logic and put into the knowledge base.
However, the more things we put in the knowledge base the longer the query will take, so we want to stick with only the essentials.
For example, we know that there can only be one WUMPUS and that each tile (except the start) has a 20% chance of being a PIT.
But the 20% chance is difficult to encode in propositional logic and not really relevant to our query, since we only care about things we know for sure.

So what do we put in the knowledge base?
We want to look for safe squares, so wee need to put a sentence for each tile defining what it means for that tile to be safe (there is no WUMPUS and no PIT there).
We also need some way to figure out whether there is a WUMPUS on a tile, so we need to add that having a WUMPUS on a tile means that the surrounding tiles all have stench.
The same goes for PIT and breeze.
Of course we also know some things about the starting tile.

Then we need a way to add new knowledge into the knowledge base based on the percepts that are encountered.

So now we need to create two functions.
One that returns the knowledge base about all relevant things we know at the start of each game, and one that given the percepts, the agent's position when those percepts were recieved, and a knowledge base, adds new relevant knowledge to the knowledge base.
A skeleton for each function is provided below

*Hint*: Remember that each sentence needs to be created for each tile and that each tile needs its own versions of all symbols. Try to think of a systematic way to name the symbols based on their tile coordinates.

In [ ]:
# --- Helper functions for propositional symbols ---

def P_sym(x, y):
    """Propositional symbol: there is a PIT at (x,y)."""
    return f"P_{x}_{y}"

def W_sym(x, y):
    """Propositional symbol: there is a WUMPUS at (x,y)."""
    return f"W_{x}_{y}"

def B_sym(x, y):
    """Propositional symbol: there is a BREEZE at (x,y)."""
    return f"B_{x}_{y}"

def S_sym(x, y):
    """Propositional symbol: there is a STENCH at (x,y)."""
    return f"S_{x}_{y}"

def Safe_sym(x, y):
    """Propositional symbol: square (x,y) is SAFE (no pit and no wumpus)."""
    return f"Safe_{x}_{y}"


def big_and(sentences):
    """Build a conjunction A1 & A2 & ... using chain + And."""
    if len(sentences) == 1:
        return sentences[0]
    return chain(And, sentences)




# --- FULL CORRECT initialWumpusKB() WITH INVERSE RULES ADDED ---


def initialWumpusKB():
    KB = set()

    # Safe <-> no pit and no wumpus
    for x in range(4):
        for y in range(4):
            KB.add(
                Equals(
                    Safe_sym(x, y),
                    And(Not(P_sym(x, y)), Not(W_sym(x, y)))
                )
            )

    # Rules linking pits/wumpus to breeze/stench + inverse rules
    for x in range(4):
        for y in range(4):
            neighbors = adjacent_tiles(x, y)
            if not neighbors:
                continue

            # PIT => all neighbors have breeze
            KB.add(
                Implies(
                    P_sym(x, y),
                    big_and([B_sym(nx, ny) for (nx, ny) in neighbors])
                )
            )

            # WUMPUS => all neighbors have stench
            KB.add(
                Implies(
                    W_sym(x, y),
                    big_and([S_sym(nx, ny) for (nx, ny) in neighbors])
                )
            )

            # *** INVERSE RULES (ABSOLUTELY REQUIRED) ***
            # No breeze => all neighbors have no pit
            KB.add(
                Implies(
                    Not(B_sym(x, y)),
                    big_and([Not(P_sym(nx, ny)) for (nx, ny) in neighbors])
                )
            )

            # No stench => all neighbors have no wumpus
            KB.add(
                Implies(
                    Not(S_sym(x, y)),
                    big_and([Not(W_sym(nx, ny)) for (nx, ny) in neighbors])
                )
            )

    # Start tile is always safe
    KB.add(Not(P_sym(0, 0)))
    KB.add(Not(W_sym(0, 0)))
    KB.add(Safe_sym(0, 0))

    return KB



# --- Add percept-based knowledge each turn ---


def addWumpusKnowledge(percepts, agent_pos, KB):
    '''
    Updates the knowledge base based on current percepts and position.
    '''
    stench, breeze, glitter, bump, scream = percepts
    x, y = agent_pos

    # 1. STENCH or ~STENCH at (x,y)
    KB.add(S_sym(x, y) if stench else Not(S_sym(x, y)))

    # 2. BREEZE or ~BREEZE at (x,y)
    KB.add(B_sym(x, y) if breeze else Not(B_sym(x, y)))

    # 3. The agent is alive → no pit & no wumpus in this tile → safe
    KB.add(Not(P_sym(x, y)))
    KB.add(Not(W_sym(x, y)))
    KB.add(Safe_sym(x, y))

    # Note: We do not encode glitter/bump/scream in KB here (as per the lab instructions)



# --- Create the set of all propositional variables used in the Wumpus World ---

wumpus_variables = set()

for x in range(4):
    for y in range(4):
        wumpus_variables.add(f"P_{x}_{y}")      # pit
        wumpus_variables.add(f"W_{x}_{y}")      # wumpus
        wumpus_variables.add(f"B_{x}_{y}")      # breeze
        wumpus_variables.add(f"S_{x}_{y}")      # stench
        wumpus_variables.add(f"Safe_{x}_{y}")   # safe cell indicator

# Helper: logical entailment using the Lab 1 query() function
def entails_true(KB, sentence):
    """
    Return True iff KB |= sentence.
    """
    true_models, false_models, none_models = query(sentence, KB, wumpus_variables)
    # Entailment: no model where sentence is False/None and at least one True model
    return len(false_models) == 0 and len(none_models) == 0 and len(true_models) > 0


## Creating the agent

With knowledge base in hand it's time to create our simple naive agent.

Reading the description of the agent ahead, you probably noticed that it requires a search algorithm to find a path from one point to another.
Since search algorithms aren't covered in this course a simple, implementation of the shortest path algorithm **A\*** has been implemented in the next code block for you to use.
It takes the start tile, goal tile, and a list of all safe tiles and then returns the next tile to walk to.
Even though search-algorithms like **A\*** are not covered in this course, you can read more about them in Chapter 3 (Solving Problems by Searching) of the course book (Artificial Intelligence: A Modern Approach).

You also recieve a function to run the game given a class to return actions (an agent), and a skeleton for the agent you will implement based on the earlier description.

In [95]:
# An implementation of a priority queue that is necessary for calculating A*
import heapq

def a_star_on_tiles(start, goal, safe_tiles):
    '''
    Takes a start tile, end tile, and a list of safe tiles and returns the next step along the shortest path to the goal.
    Assumes that the world is a grid where each tile is connected to the adjacent tiles (not diagonally)
    Args:
        start (int, int): The coordinates of the start tile
        goal (int, int): The tile to find a shortest path towards
        safe_tiles [(int, int)]: A list of the tiles that can be traversed (should include start and goal)
    Returns ((int, int)/None): The next tile in the shortest path if one exists
    '''
    frontier = []
    heapq.heappush(frontier, (0, start))
    came_from = {}
    cost_so_far = {}
    came_from[start] = None
    cost_so_far[start] = 0
    
    # Run A* to get a dictionary containing for each tile, the previous tile along the path
    while len(frontier) > 0:
        distance, (x, y) = heapq.heappop(frontier)
        
        if (x,y) == goal:
            break
        
        for neighbor in [n for n in adjacent_tiles(x,y) if n in safe_tiles]:
            new_cost = cost_so_far[(x,y)] + 1 #Can be adapted to handle turning time
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                priority = new_cost + abs(x-neighbor[0]) + abs(y-neighbor[1])
                heapq.heappush(frontier, (priority, neighbor))
                came_from[neighbor] = (x,y)
    
    # Walk backward along the path until the first tile beyond the start is found
    # If the start tile is unreachable, return None
    current = goal
    while True:
        if current not in came_from:
            return None
        if came_from[current] == start:
            break
        current = came_from[current]
    
    return current

In [96]:
# --- Pretty print the Wumpus World ---

def show_world_pretty(w, step=None):
    """
    Prints the Wumpus World in a clean 4x4 grid with cell borders.
    Much easier to see what happens each step.
    """
    if step is not None:
        print(f"\nStep: {step}, Performance: {w.performance}")
    else:
        print(f"\nPerformance: {w.performance}")

    print("+----+----+----+----+")
    for y in range(3, -1, -1):  # top row first
        row = "|"
        for x in range(4):
            contents = "".join(w.cave[x][y])
            if contents == "":
                contents = " "   # empty cell
            row += f"{contents:^4}|"
        print(row)
        print("+----+----+----+----+")


In [98]:
# --- Main game loop function ---

def play_game(agent, show=True):
    '''
    Initiates and runs a random Wumpus World by calling the Agent.act() function to get actions
    Args:
        agent (any): An object which has implemented an act function
        show (bool): Whether to print the states of the game or not
    '''
    w = WumpusWorld()
    step = 0

    show_world_pretty(w, step)

    while not w.game_over:
        percepts = w.perceive()
        action = agent.act(percepts, w.agent_pos, w.agent_dir)
        w.actuate(action)
        step += 1
        show_world_pretty(w, step)
        # debug internal logic:
        print("Percepts:", percepts, "| Action:", action)

    print("\nGAME OVER")
    print(f"Final performance: {w.performance}")

In [99]:
class SimpleAgent():
    '''
    Simple knowledge-based agent for the Wumpus World.

    - Maintains a propositional KB using initialWumpusKB() and addWumpusKnowledge().
    - In addition, keeps its own sets of "safe" and "visited" tiles for fast planning.
    - Uses a_star_on_tiles() to move between safe tiles.
    '''

    def __init__(self):
        # Full propositional KB
        self.KB = initialWumpusKB()

        # Tiles that the agent believes to be safe (for movement planning)
        self.safe = set()
        self.safe.add((0, 0))   # start is known safe

        # Tiles already visited
        self.visited = set()

        # Does the agent have the gold?
        self.has_gold = False

    # ---------- Helper methods ----------

    def _neighbors(self, pos):
        """Return all valid neighboring tiles of pos."""
        x, y = pos
        return adjacent_tiles(x, y)

    def _mark_safe_from_percepts(self, position, stench, breeze):
        """
        Local reasoning:
        - The current tile is safe (because we are alive)
        - If there is NO stench and NO breeze, then all neighboring tiles are safe
        """
        self.safe.add(position)
        if not stench and not breeze:
            for n in self._neighbors(position):
                self.safe.add(n)

    def _desired_direction(self, position, target):
        """Return direction (RIGHT/DOWN/LEFT/UP) to face from position to target."""
        x, y = position
        tx, ty = target
        dx, dy = tx - x, ty - y

        if dx == 1 and dy == 0:
            return RIGHT
        if dx == -1 and dy == 0:
            return LEFT
        if dx == 0 and dy == 1:
            return UP
        if dx == 0 and dy == -1:
            return DOWN
        return None  # not adjacent

    # ---------- Main decision function ----------

    def act(self, percepts, position, direction):
        '''
        Decide the next action based on percepts, position and direction.
        Args:
            percepts ([bool]): [stench, breeze, glitter, bump, scream]
            position (int, int): current tile (x,y)
            direction (int): current facing direction
        '''
        stench, breeze, glitter, bump, scream = percepts
        self.visited.add(position)

        # 1) Update the logical KB with new percepts
        addWumpusKnowledge(percepts, position, self.KB)

        # 2) Update our fast "safe" set from local percepts
        self._mark_safe_from_percepts(position, stench, breeze)

        # 3) If we see glitter and don't have gold yet -> GRAB
        if glitter and not self.has_gold:
            self.has_gold = True
            return GRAB

        # 4) If we have gold and are at the start -> CLIMB
        if self.has_gold and position == (0, 0):
            return CLIMB

        # 5) Determine safe tiles and unvisited safe tiles
        safe_tiles = list(self.safe)
        unvisited_safe = [t for t in safe_tiles if t not in self.visited]

        # 6) Choose a goal:
        #    - if there are unvisited safe tiles, go to one of them;
        #    - otherwise go to start (0,0) and then climb.
        if unvisited_safe:
            goal = unvisited_safe[0]
        else:
            goal = (0, 0)

        # 7) If we are already at the goal
        if position == goal:
            # If goal is start and we either have gold or no more moves -> CLIMB
            if position == (0, 0):
                return CLIMB
            return NONE

        # 8) Use A* to find next tile along safe path
        next_tile = a_star_on_tiles(position, goal, safe_tiles)

        # If no path exists:
        if next_tile is None:
            # If stuck at start -> give up and climb
            if position == (0, 0):
                return CLIMB
            # Otherwise, mark goal as visited to avoid choosing it again
            self.visited.add(goal)
            return NONE

        # 9) If we are not facing the next tile, turn (minimal turn)
        desired_dir = self._desired_direction(position, next_tile)
        if desired_dir is None:
            return NONE  # should not happen

        if direction != desired_dir:
            # Choose shortest turn (left vs right)
            if (direction - desired_dir) % 4 == 1:
                return TURN_LEFT
            else:
                return TURN_RIGHT

        # 10) Facing the correct direction -> move forward
        return FORWARD


In [114]:
# Execute this cell to play one game over the Wumpus World with the SimpleAgent

play_game(SimpleAgent())


Step: 0, Performance: 0
+----+----+----+----+
| p  |    |    | w  |
+----+----+----+----+
| g  | p  |    |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
| a  |    |    | p  |
+----+----+----+----+

Step: 1, Performance: -1
+----+----+----+----+
| p  |    |    | w  |
+----+----+----+----+
| g  | p  |    |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p  |
+----+----+----+----+
Percepts: (False, False, False, False, False) | Action: forward

Step: 2, Performance: -2
+----+----+----+----+
| p  |    |    | w  |
+----+----+----+----+
| g  | p  |    |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p  |
+----+----+----+----+
Percepts: (False, False, False, False, False) | Action: turn_right

Step: 3, Performance: -3
+----+----+----+----+
| p  |    |    | w  |
+----+----+----+----+
| g  | p  |    |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p

In this run, the agent explored the cave tile by tile. Every time it moved or turned, it lost one point, so the score slowly went down at the beginning.

When the agent reached the gold (Step 19), nothing special happened yet.
At Step 20, it performed GRAB, which picked up the gold. This also cost one point.

After grabbing the gold, the agent tried to return to the starting square (0,0). Each action on the way back continued to subtract one point.

Finally, at Step 25, the agent arrived at the start with the gold in its inventory and executed CLIMB. This ends the game. CLIMB itself costs one point, but because the agent escaped with the gold, it receives a reward of +1000 points.

So the final score becomes the small negative total from all the actions, plus the big reward for successfully bringing the gold home. That's why the game ends with a high final performance of 975.

In [116]:
# Test the game again

play_game(SimpleAgent())


Step: 0, Performance: 0
+----+----+----+----+
|    | g  |    |    |
+----+----+----+----+
|    |    | w  |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
| a  |    |    | p  |
+----+----+----+----+

Step: 1, Performance: -1
+----+----+----+----+
|    | g  |    |    |
+----+----+----+----+
|    |    | w  |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p  |
+----+----+----+----+
Percepts: (False, False, False, False, False) | Action: forward

Step: 2, Performance: -2
+----+----+----+----+
|    | g  |    |    |
+----+----+----+----+
|    |    | w  |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p  |
+----+----+----+----+
Percepts: (False, False, False, False, False) | Action: turn_right

Step: 3, Performance: -3
+----+----+----+----+
|    | g  |    |    |
+----+----+----+----+
|    |    | w  |    |
+----+----+----+----+
|    |    | p  |    |
+----+----+----+----+
|    | a  |    | p